# Batching Strategies for LLM Inference

In notebook 03 we saw that batching is critical for GPU utilisation — processing
multiple requests together amortises the cost of loading weights from memory.

But LLM inference has a unique challenge: **requests arrive at different times,
have different prompt lengths, and finish at different times**. Naive batching
wastes enormous GPU cycles waiting for the slowest request.

This notebook covers the evolution of batching strategies:
1. Static batching — simple but wasteful
2. Dynamic batching — better, still limited
3. Continuous batching (iteration-level) — the modern standard
4. Splitfuse / chunked prefill — disaggregating prefill and decode
5. Comparison and tradeoffs

## 1. Static batching — the naive approach

Collect N requests, pad them to the same length, process as a single batch.
Wait until ALL requests in the batch finish generating before returning any.

```
Time ──────────────────────────────────────────────────────────►

Req A: [prompt████|gen████████████|DONE|pad pad pad pad pad pad]
Req B: [prompt████████████|gen████████████████████████████|DONE]
Req C: [prompt██|gen██████████████████|DONE|pad pad pad pad pad]
        ◄── batch starts            all must wait for B ──►
```

### Problems:
- **Padding waste**: shorter sequences burn compute on pad tokens
- **Head-of-line blocking**: fast requests wait for the slowest one
- **Low utilisation**: GPU is idle while waiting for a full batch to accumulate
- **Memory waste**: must allocate KV cache for max possible sequence length for all slots

### When it's acceptable:
- Offline batch processing (all inputs known upfront)
- Fixed output length (classification, embedding)
- Very simple serving setups

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulate static batching
np.random.seed(42)

def simulate_static_batching(requests, batch_size=4, time_per_token=10):
    """Simulate static batching. Returns (completion_times, gpu_utilisation)."""
    results = []
    total_useful_work = 0
    total_time = 0
    
    for batch_start in range(0, len(requests), batch_size):
        batch = requests[batch_start:batch_start + batch_size]
        prompt_lens = [r["prompt_len"] for r in batch]
        output_lens = [r["output_len"] for r in batch]
        
        # Must pad to max lengths
        max_total = max(p + o for p, o in zip(prompt_lens, output_lens))
        batch_time = max_total * time_per_token
        
        for i, r in enumerate(batch):
            actual_time = (r["prompt_len"] + r["output_len"]) * time_per_token
            total_useful_work += actual_time
            results.append({
                "request_id": batch_start + i,
                "start_time": total_time,
                "finish_time": total_time + batch_time,  # waits for longest
                "actual_work": actual_time,
                "wasted": batch_time - actual_time,
            })
        
        total_time += batch_time
    
    utilisation = total_useful_work / (total_time * batch_size) * 100
    return results, utilisation, total_time

# Create varied requests
requests = [
    {"prompt_len": np.random.randint(10, 100), "output_len": np.random.randint(20, 200)}
    for _ in range(16)
]

static_results, static_util, static_total = simulate_static_batching(requests)
print(f"Static batching:")
print(f"  Total wall time: {static_total} ms")
print(f"  GPU utilisation: {static_util:.1f}%")
print(f"  Avg wait (after done): {np.mean([r['wasted'] for r in static_results]):.0f} ms")

## 2. Dynamic batching — group arrivals by deadline

Instead of a fixed batch, collect requests that arrive within a small time window
(e.g. 50ms) and batch them together. Still pads and still blocks.

```
                    ┌─ batch window (50ms) ─┐
Arrivals:   A  B      C  D  E              F  G
            └─batch 1──┘  └──batch 2──┘    └─batch 3─┘
```

### Improvement over static:
- Doesn't wait forever for a full batch — uses a timeout
- Can group similar-length prompts together

### Still has:
- Head-of-line blocking within each batch
- Padding waste
- Slots freed only when entire batch completes

Used by: TensorRT-LLM (as a baseline), Triton Inference Server

## 3. Continuous batching — the modern standard

The key insight: **don't wait for the whole batch to finish. When one request
finishes, immediately replace it with a new one.**

Each iteration (one decode step), the scheduler decides which requests are active:

```
Iteration 1: [A, B, C, D]     ← full batch
Iteration 2: [A, B, C, D]     ← all still generating
Iteration 3: [A, B, C, D]     ← A finishes!
Iteration 4: [E, B, C, D]     ← E immediately takes A's slot
Iteration 5: [E, B, C, D]     ← C finishes!
Iteration 6: [E, B, F, D]     ← F takes C's slot
   ...
```

### Advantages:
- **No head-of-line blocking** — finished requests return immediately
- **No padding** — each request only processes its own tokens
- **Higher GPU utilisation** — batch slots are always occupied
- **Lower latency** — new requests start as soon as a slot opens

### Implementation:
- vLLM (PagedAttention + continuous batching)
- TGI (Hugging Face Text Generation Inference)
- TensorRT-LLM (in-flight batching)
- SGLang

In [ ]:
def simulate_continuous_batching(requests, max_batch_size=4, time_per_token=10):
    """Simulate continuous batching. New requests fill slots as others finish."""
    results = []
    # Track active slots: each has remaining tokens to generate
    active = []  # list of {request_id, remaining, start_time}
    queue = list(range(len(requests)))
    current_time = 0
    total_useful_steps = 0
    total_steps = 0
    
    while active or queue:
        # Fill empty slots from queue
        while len(active) < max_batch_size and queue:
            req_id = queue.pop(0)
            r = requests[req_id]
            active.append({
                "request_id": req_id,
                "remaining": r["prompt_len"] + r["output_len"],
                "start_time": current_time,
            })
        
        if not active:
            break
        
        # One iteration: process one token for each active request
        total_steps += max_batch_size  # GPU capacity
        total_useful_steps += len(active)  # actual work
        
        finished = []
        for slot in active:
            slot["remaining"] -= 1
            if slot["remaining"] <= 0:
                finished.append(slot)
        
        current_time += time_per_token
        
        for slot in finished:
            active.remove(slot)
            results.append({
                "request_id": slot["request_id"],
                "start_time": slot["start_time"],
                "finish_time": current_time,
                "actual_work": (requests[slot["request_id"]]["prompt_len"] + 
                               requests[slot["request_id"]]["output_len"]) * time_per_token,
                "wasted": 0,  # no waiting!
            })
    
    utilisation = total_useful_steps / total_steps * 100 if total_steps > 0 else 0
    return results, utilisation, current_time

continuous_results, continuous_util, continuous_total = simulate_continuous_batching(requests)

print(f"Continuous batching:")
print(f"  Total wall time: {continuous_total} ms")
print(f"  GPU utilisation: {continuous_util:.1f}%")
print(f"  Avg latency: {np.mean([r['finish_time'] - r['start_time'] for r in continuous_results]):.0f} ms")
print(f"\nvs Static batching:")
print(f"  Total wall time: {static_total} ms")
print(f"  GPU utilisation: {static_util:.1f}%")
print(f"  Avg latency: {np.mean([r['finish_time'] - r['start_time'] for r in static_results]):.0f} ms")
print(f"\nContinuous batching: {static_total/continuous_total:.1f}x faster wall time, {continuous_util/static_util:.1f}x better utilisation")

In [ ]:
# Visualise the two approaches
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

colors = plt.cm.tab20(np.linspace(0, 1, len(requests)))

# Static batching
for r in static_results:
    ax1.barh(r["request_id"], r["actual_work"], left=r["start_time"],
             color=colors[r["request_id"]], edgecolor='black', linewidth=0.5)
    ax1.barh(r["request_id"], r["wasted"], left=r["start_time"] + r["actual_work"],
             color='lightgray', edgecolor='black', linewidth=0.5, alpha=0.5)

ax1.set_ylabel("Request ID")
ax1.set_title(f"Static batching — {static_util:.0f}% utilisation, {static_total} ms total")
ax1.legend(handles=[
    mpatches.Patch(color='tab:blue', label='Useful work'),
    mpatches.Patch(color='lightgray', alpha=0.5, label='Wasted (waiting)'),
], loc='upper right')

# Continuous batching
for r in continuous_results:
    duration = r["finish_time"] - r["start_time"]
    ax2.barh(r["request_id"], duration, left=r["start_time"],
             color=colors[r["request_id"]], edgecolor='black', linewidth=0.5)

ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Request ID")
ax2.set_title(f"Continuous batching — {continuous_util:.0f}% utilisation, {continuous_total} ms total")

plt.tight_layout()
plt.show()

## 4. Chunked prefill (Splitfuse) — handling the prefill/decode imbalance

Continuous batching has one remaining problem: **prefill and decode have very
different compute profiles**.

When a new long-prompt request joins the batch, its prefill is compute-heavy and
blocks the decode steps of all other active requests:

```
Without chunked prefill:
──────────────────────────────────────────────
Iteration 5: [decode A, decode B, PREFILL_NEW_REQUEST (2000 tokens)]
             ─────────── takes 100ms ────────────────────────────
             A and B get no tokens for 100ms!

With chunked prefill:
──────────────────────────────────────────────
Iteration 5: [decode A, decode B, prefill_chunk_1 (128 tokens)]
Iteration 6: [decode A, decode B, prefill_chunk_2 (128 tokens)]
Iteration 7: [decode A, decode B, prefill_chunk_3 (128 tokens)]
   ...         A and B still get tokens every ~5ms
```

### How it works:
- Long prompts are split into fixed-size chunks (e.g. 128 or 512 tokens)
- Each iteration processes: decode tokens (1 per request) + one prefill chunk
- The prefill chunks build up the KV cache incrementally
- Active decode requests maintain consistent inter-token latency

### Tradeoff:
- Slightly longer time-to-first-token (TTFT) for the new request
- But existing requests maintain consistent streaming speed
- Better tail latency across all requests

Used by: vLLM (chunked prefill), SGLang, Sarathi

In [ ]:
# Simulate the impact of a large prefill on decode latency

def simulate_prefill_impact(prefill_length, chunk_size=None, decode_batch=4, time_per_token=5):
    """
    Simulate inter-token latency for decode requests when a large prefill arrives.
    chunk_size=None means monolithic prefill (no chunking).
    """
    decode_latencies = []
    
    if chunk_size is None:
        # Monolithic prefill: one big stall
        prefill_time = prefill_length * 0.5  # prefill tokens are ~0.5ms each (parallel)
        # During prefill, decode requests stall
        decode_latencies.append(prefill_time)  # one big gap
        # Then normal decode resumes
        for _ in range(19):
            decode_latencies.append(time_per_token)
    else:
        # Chunked: interleave prefill chunks with decode steps
        chunks_needed = (prefill_length + chunk_size - 1) // chunk_size
        chunk_time = chunk_size * 0.5  # time for one chunk
        
        for i in range(20):
            if i < chunks_needed:
                # Decode + prefill chunk in same iteration
                decode_latencies.append(time_per_token + chunk_time)
            else:
                decode_latencies.append(time_per_token)
    
    return decode_latencies

prefill_len = 2000  # new request with 2000 token prompt

latencies_no_chunk = simulate_prefill_impact(prefill_len, chunk_size=None)
latencies_chunk_128 = simulate_prefill_impact(prefill_len, chunk_size=128)
latencies_chunk_512 = simulate_prefill_impact(prefill_len, chunk_size=512)

fig, ax = plt.subplots(figsize=(12, 4))
x = range(20)
ax.plot(x, latencies_no_chunk, 'ro-', label="No chunking (monolithic prefill)", linewidth=2)
ax.plot(x, latencies_chunk_512, 'bs-', label="Chunk size = 512", linewidth=2)
ax.plot(x, latencies_chunk_128, 'g^-', label="Chunk size = 128", linewidth=2)
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.5, label="Ideal decode latency")
ax.set_xlabel("Decode iteration")
ax.set_ylabel("Inter-token latency (ms)")
ax.set_title(f"Impact of {prefill_len}-token prefill on decode latency")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(latencies_no_chunk) * 1.1)
plt.tight_layout()
plt.show()

print(f"Monolithic prefill: decode stalls for {latencies_no_chunk[0]:.0f} ms (p99 latency spike)")
print(f"Chunked (512): max decode latency = {max(latencies_chunk_512):.0f} ms")
print(f"Chunked (128): max decode latency = {max(latencies_chunk_128):.0f} ms")

## 5. Comparison and tradeoffs

| Strategy | Throughput | Latency | Complexity | When to use |
|----------|-----------|---------|-----------|------------|
| Static | Low | High (blocking) | Simple | Offline batch jobs |
| Dynamic | Medium | Medium | Medium | Low-traffic APIs |
| Continuous | High | Low | High | Production serving |
| Chunked prefill | Highest | Lowest p99 | Highest | Long-context serving |

### The key metrics for serving:

- **Throughput** (tokens/sec): how much total work the system does
- **Time to first token (TTFT)**: how long before the user sees output start
- **Inter-token latency (ITL)**: the gap between consecutive tokens (streaming smoothness)
- **p99 latency**: worst-case experience (tail latency)

Continuous batching optimises throughput and average latency.
Chunked prefill additionally tames tail latency.

In [ ]:
# Summary comparison across strategies
print("=" * 70)
print(f"{'Strategy':<22} {'Wall time':<12} {'Utilisation':<14} {'Avg latency':<14} {'Wasted time':<12}")
print("=" * 70)

# Static
avg_latency_static = np.mean([r['finish_time'] - r['start_time'] for r in static_results])
avg_waste_static = np.mean([r['wasted'] for r in static_results])
print(f"{'Static':<22} {static_total:<12} {static_util:<14.1f}% {avg_latency_static:<14.0f}ms {avg_waste_static:<12.0f}ms")

# Continuous
avg_latency_cont = np.mean([r['finish_time'] - r['start_time'] for r in continuous_results])
print(f"{'Continuous':<22} {continuous_total:<12} {continuous_util:<14.1f}% {avg_latency_cont:<14.0f}ms {'0':<12}ms")

print("=" * 70)
print(f"\nContinuous batching is {static_total/continuous_total:.1f}x faster with")
print(f"{continuous_util/static_util:.1f}x better GPU utilisation.")

## Key takeaways for LLM serving

1. **Static batching is dead** for online serving — head-of-line blocking is unacceptable
   when users are streaming tokens

2. **Continuous batching** is the minimum standard — vLLM, TGI, and TensorRT-LLM all
   implement it. Requests start and finish independently.

3. **Chunked prefill** is the next evolution — critical when you have mixed workloads
   (some short requests, some long-context RAG queries). Without it, a 100K-token
   prefill blocks all active streams.

4. **Memory management** (next notebook) is what makes continuous batching practical —
   PagedAttention lets you dynamically allocate and free KV cache memory per-request
   without fragmentation.

5. The ultimate goal: **keep every GPU cycle doing useful work**. Idle slots = wasted money.